# 03 — Evaluation Dashboard
Run all metrics and display a comprehensive evaluation of generated question sets.

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

sys.path.insert(0, str(Path("..").resolve()))
from eval.metrics import evaluate

GENERATED = Path("../dataset/generated")
RESULTS   = Path("../eval/results")
RESULTS.mkdir(exist_ok=True)

## Run evaluation on all files

In [ ]:
reports = []
for f in sorted(GENERATED.glob("*.json")):
    questions = json.loads(f.read_text(encoding="utf-8"))
    if not questions:
        continue
    report = evaluate(questions)
    report["file"]      = f.name
    report["game_type"] = questions[0].get("game_type", "?")
    report["source"]    = questions[0].get("source_doc", f.stem)
    reports.append(report)
    status = "✓ PASS" if report["pass"] else "✗ FAIL"
    print(f"{status}  {f.name:<45}  diversity={report['diversity_score']}  balance={report['balance_score']}")

print(f"\nTotal: {sum(r['pass'] for r in reports)}/{len(reports)} passed")

## Metrics overview table

In [ ]:
rows = []
for r in reports:
    rows.append({
        "File":       r["file"],
        "Game":       r["game_type"],
        "N":          r["n_questions"],
        "Balance":    r["balance_score"],
        "Diversity":  r["diversity_score"],
        "Relevance":  r.get("relevance_score", "-"),
        "Fmt OK":     r["format_validity"]["pass_rate"],
        "Pass":       "✓" if r["pass"] else "✗",
    })
pd.DataFrame(rows).set_index("File")

## Radar chart — metric comparison per game type

In [ ]:
import numpy as np

game_reports = {}
for r in reports:
    g = r["game_type"]
    if g not in game_reports:
        game_reports[g] = {"balance": [], "diversity": [], "fmt": []}
    game_reports[g]["balance"].append(r["balance_score"])
    game_reports[g]["diversity"].append(r["diversity_score"])
    game_reports[g]["fmt"].append(r["format_validity"]["pass_rate"])

games = list(game_reports.keys())
categories = ["Balance", "Diversity", "Format"]
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
colors = ["#336600", "#f5c518", "#3b82f6", "#a855f7", "#ef4444"]

for idx, (game, color) in enumerate(zip(games, colors)):
    vals = [
        np.mean(game_reports[game]["balance"]),
        np.mean(game_reports[game]["diversity"]),
        np.mean(game_reports[game]["fmt"]),
    ]
    vals += vals[:1]
    ax.plot(angles, vals, "o-", linewidth=2, label=game, color=color)
    ax.fill(angles, vals, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=12)
ax.set_ylim(0, 1)
ax.set_title("Metric Radar by Game Type", size=14, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig(RESULTS / "radar_chart.png", bbox_inches="tight")
plt.show()
print("Saved: eval/results/radar_chart.png")

## Save full report

In [ ]:
from datetime import datetime

out = RESULTS / f"eval_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
out.write_text(json.dumps({"timestamp": datetime.now().isoformat(), "reports": reports},
                            ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Full report saved → {out}")